In [2]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import xarray as xr
import rioxarray
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import numpy as np
import regionmask
import matplotlib.patheffects as pe
import textwrap

In [3]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins_NAtl.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [4]:
# open daily mean files
ds = xr.open_dataset("datasets/ERA5_1940/sst_dailyMean_ERA5_1940-2025")

# rename columns
ds = ds.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

print(ds)

<xarray.Dataset> Size: 27GB
Dimensions:  (time: 15738, lat: 361, lon: 1201)
Coordinates:
  * time     (time) datetime64[ns] 126kB 1940-06-01 1940-06-02 ... 2025-11-30
  * lat      (lat) float64 3kB 90.0 89.75 89.5 89.25 89.0 ... 0.75 0.5 0.25 0.0
  * lon      (lon) float64 10kB -120.0 -119.8 -119.5 ... 179.5 179.8 180.0
    number   int64 8B ...
    expver   (time) <U4 252kB ...
Data variables:
    sst      (time, lat, lon) float32 27GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-19T23:33 GRIB to CDM+CF via cfgrib-0.9.1...


In [5]:
# filter to variable
sst = ds["sst"]

# convert lon to -180-180
sst = sst.assign_coords(
    lon=(((sst.lon + 180) % 360) - 180)
).sortby("lon")

print(sst)

<xarray.DataArray 'sst' (time: 15738, lat: 361, lon: 1201)> Size: 27GB
[6823383018 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 126kB 1940-06-01 1940-06-02 ... 2025-11-30
  * lat      (lat) float64 3kB 90.0 89.75 89.5 89.25 89.0 ... 0.75 0.5 0.25 0.0
  * lon      (lon) float64 10kB -180.0 -120.0 -119.8 ... 179.2 179.5 179.8
    number   int64 8B ...
    expver   (time) <U4 252kB ...
Attributes: (12/31)
    GRIB_paramId:                             34
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      433561
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           sst
    GRIB_units:                               K
    long_name:                                Sea surface temperature
    units:                             

In [ ]:
# # add CRS and spatial dims
# sst = sst.rio.write_crs("EPSG:4326")
# sst = sst.rio.set_spatial_dims(x_dim="lon", y_dim="lat")

# # filter to N Atlantic basin
# region = basins[basins["basin name"] == "N Atlantic"]
# sst_filt = (
#     sst.rio.clip(region.geometry, region.crs, drop=True)
# )

In [ ]:
# # calculate the mean SST (not for anomaly calc)
# daily_mean = sst.resample(time="1D").mean()